# Protocol 4 — Engineer features: parts, splits, and numerical (embeddings/structure)

**Feature engineering is the stage between raw sequences and the CPP signature.** Before [Comparative Physicochemical Profiling (CPP)](https://github.com/breimanntools/aaanalysis) can contrast two groups, every protein must be turned into a structured feature space. AAanalysis builds that space from one grammar with three axes: **Part** (*where* in the sequence), **Split** (*how* the positions are selected), and a value axis that is either a physicochemical **Scale** (sequence arm) or a per-residue numerical **dimension** (numerical arm). This protocol covers both arms.

This protocol is part of the [Protocols catalog (#35)](https://github.com/breimanntools/aaanalysis/issues/35). It follows the standard pipeline-chained structure: *When to use it -> Input -> Run -> Output -> How to interpret -> Common mistakes -> Next step.*

## When to use it

Use this protocol after you have **labelled sequences** (from a sampling step or a curated `df_seq`) and before you run the CPP signature itself. It answers the engineering questions that sit underneath every CPP analysis:

- **How do I encode *where*?** Slice each protein into TMD-centric **parts**. By default `get_df_parts` returns the part `tmd` (a base part) plus the composites `jmd_n_tmd_n` and `tmd_c_jmd_c` (built from the base parts `jmd_n`/`tmd`/`jmd_c`).
- **How do I encode *which positions*?** Build a **split** configuration (Segment / Pattern / PeriodicPattern).
- **What value do I average?** Either a physicochemical **scale** (the *sequence arm*, consumed by `CPP.run`) or a per-residue numerical vector such as a protein-language-model embedding or a structural channel (the *numerical arm*, consumed by `CPP.run_num`).

The two arms share the identical Part-Split grammar and produce the identical `df_feat` schema. Only the value source differs, so anything you learn here transfers directly to embeddings and structure.

## Input

A `df_seq` with one row per protein, an `entry` identifier, a `sequence`, and a binary `label` column (test class = 1 vs. reference class = 0). For a **domain-level** task it also carries `tmd_start` / `tmd_stop`, from which the part boundaries are derived. Upstream this `df_seq` typically comes from *Protocol 3 - Sampling* or from a curated labelled set.

The **numerical arm** adds one more input: a `dict_num` mapping each `entry` to a per-residue tensor of shape `(L, D)`, where `L` equals the sequence length and `D` is the number of numerical dimensions (embedding/structure channels). Here we use the bundled `DOM_GSEC` gamma-secretase dataset, and for the numerical arm we build a **small synthetic** `dict_num` with NumPy so the protocol runs offline (no model download). The real workflow would substitute a ProtT5/ESM embedding or a DSSP/structure tensor of the same shape.

In [1]:
import aaanalysis as aa
import numpy as np

aa.options["verbose"] = False
aa.options["random_state"] = 42

# Domain-level gamma-secretase data: substrates (label=1) vs non-substrates (0).
# load_dataset(name=..., n=10) returns 2*n = 20 rows (10 per class).
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()          # use the label column, NOT len(df_seq)
df_seq.head()

,entry,sequence,label,tmd_start,tmd_stop,jmd_n,tmd,jmd_c
0,Q14802,MQKVTLGLLVFLAGFPVLDANDLEDKNSPFYYDWHSLQVGGLICAG...,0,37,59,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS
1,Q86UE4,MAARSWQDELAQQAEEGSARLREMLSVGLGFLRTELGLDLGLEPKR...,0,50,72,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR
2,Q969W9,MHRLMGVNSTAAAAAGQPNVSCTCNCKRSLFQSMEITELEFVQIII...,0,41,63,FQSMEITELE,FVQIIIIVVVMMVMVVVITCLLS,HYKLSARSFI
3,P53801,MAPGVARGPTPYWRLRLGGAALLLLLIPVAAAQEPPGAACSQNTNK...,0,97,119,RWGVCWVNFE,ALIITMSVVGGTLLLGIAICCCC,CCRRKRSRKP
4,Q8IUW5,MAPRALPGSAVLAAAVFVGGAVSSPLVAPDNGSSRTLHSRTETTPS...,0,59,81,NDTGNGHPEY,IAYALVPVFFIMGLFGVLICHLL,KKKGYRCTTE


## Run - Arm A: sequence features (scale-based)

The sequence arm turns `df_seq` into a CPP-ready feature space in three steps:

1. **`SequenceFeature.get_df_parts`** slices each protein into the TMD-centric parts (by default the base part `tmd` plus the composites `jmd_n_tmd_n` and `tmd_c_jmd_c`), producing the `df_parts` that `CPP` consumes.
2. **`SequenceFeature.get_split_kws`** builds the split configuration - one entry per split type (Segment, Pattern, PeriodicPattern).
3. **`SequenceFeature.get_features`** enumerates the full `PART-SPLIT-SCALE` feature space as a list of feature ids.

In [2]:
sf = aa.SequenceFeature()

# 1) Parts: slice sequences into TMD-centric parts
df_parts = sf.get_df_parts(df_seq=df_seq)

# 2) Split configuration: Segment / Pattern / PeriodicPattern
split_kws = sf.get_split_kws()

# Split types in the configuration: ['Segment', 'Pattern', 'PeriodicPattern']
list(split_kws)

['Segment', 'Pattern', 'PeriodicPattern']

In [3]:
# 3) Enumerate the PART-SPLIT-SCALE space. The full scale set explodes
#    combinatorially, so restrict to a 3-scale slice for the demo.
df_scales = aa.load_scales()
small_scales = list(df_scales.columns[:3])

features = sf.get_features(split_kws=split_kws, list_scales=small_scales)
# Number enumerated (3 parts x splits x 3 scales): len(features)
len(features), features[:5]

(2970,
 ['TMD-Segment(1,1)-ANDN920101',
  'TMD-Segment(1,1)-ARGP820101',
  'TMD-Segment(1,1)-ARGP820102',
  'TMD-Segment(1,2)-ANDN920101',
  'TMD-Segment(1,2)-ARGP820101'])

In [4]:
# Optional: reduce redundancy. Build the feature matrix, then drop columns
# that correlate above max_cor with an already-kept feature.
# Note: filter_correlation keeps the FIRST-SEEN representative of each
# correlated group. Applied pre-CPP on the raw PART-SPLIT-SCALE enumeration
# order, it yields a redundancy-reduced but importance-agnostic set. To keep
# the most important representative, apply it to an importance-ordered
# CPP.run / df_feat output instead (see Protocol 1 - CPP signature).
X = sf.feature_matrix(features=features, df_parts=df_parts, n_jobs=1)
is_selected = aa.NumericalFeature.filter_correlation(X, max_cor=0.7)

# Features kept after redundancy filtering:
int(is_selected.sum())

368

## Run - Arm B: numerical features (embeddings / structure)

The numerical arm replaces the per-AA scale lookup with a **per-residue tensor**. The Part-Split grammar is unchanged; only the value source differs.

1. Build a `dict_num` of `{entry: (L, D) ndarray}`. We synthesise one with `np.random.default_rng(42)`; `rng.random` already yields values in `[0, 1)`, which is exactly the range `CPP.run_num` expects (its `max_std_test` pre-filter is calibrated for `[0, 1]`), so no normalisation step is needed.
2. **`EmbeddingPreprocessor.build_scales` / `build_cat`** name the `D` dimensions (`df_scales` of shape `(20, D)`, `df_cat` of shape `(D, ...)`) so the `CPP` invariant `D == len(df_scales.columns) == len(df_cat)` holds. The per-AA scale values are *not* the per-residue value source for `run_num` (that is `dict_num_parts`), but the `df_cat` categories - derived from these pseudo-scales by clustering - **do** drive category-aware redundancy filtering and grouping. Deriving `df_scales`/`df_cat` from the *same* `dict_num` (rather than fabricating an arbitrary `(20, D)` / `df_cat`) is therefore what keeps both the invariant and the categories meaningful.
3. **`NumericalFeature.get_parts`** slices the sequence strings AND the tensor with the *same* boundaries, returning `(df_parts, dict_num_parts)`.
4. **`CPP.run_num`** runs the identical CPP algorithm on the sliced tensor.

In [5]:
D = 4                                   # tiny embedding dimensionality
rng = np.random.default_rng(42)

# Per-residue tensor: one (L, D) array per entry, L == sequence length, in [0, 1)
dict_num = {e: rng.random((len(s), D))
            for e, s in zip(df_seq["entry"], df_seq["sequence"])}

# Name the D dimensions. dict_num_parts (not these per-AA values) is the
# per-residue value source for run_num; but df_cat's categories - derived
# from these pseudo-scales - drive category-aware redundancy filtering, so we
# derive both from the same dict_num rather than fabricating them.
# Shapes: df_scales_num is (20, D); df_cat_num is (D, ...).
ep = aa.EmbeddingPreprocessor()
df_scales_num = ep.build_scales(df_seq=df_seq, dict_num=dict_num)
df_cat_num = ep.build_cat(df_scales=df_scales_num)
df_scales_num.head()

/var/folders/sv/65tlch_10198qgmpwcp6408r0000gn/T/ipykernel_27511/2680810611.py:14: UserWarning: Pseudo-scales are dataset-dependent (averaged over df_seq). For reproducible cross-dataset comparison, compute them once on a fixed reference corpus and reuse the resulting df_scales.
  df_scales_num = ep.build_scales(df_seq=df_seq, dict_num=dict_num)


,dim_0,dim_1,dim_2,dim_3
A,0.492584,0.497712,0.485069,0.513540
C,0.502098,0.506278,0.518440,0.502096
D,0.487394,0.508945,0.499097,0.498795
E,0.482504,0.494474,0.511045,0.497436
F,0.508067,0.473032,0.475938,0.501151


In [6]:
# Slice sequences AND tensor with shared part boundaries
df_parts_num, dict_num_parts = aa.NumericalFeature.get_parts(df_seq=df_seq, dict_num=dict_num)

# Each part tensor has shape (n_samples, L_part_max, D), NaN-padded
{k: v.shape for k, v in dict_num_parts.items()}

{'tmd': (20, 23, 4), 'jmd_n_tmd_n': (20, 22, 4), 'tmd_c_jmd_c': (20, 21, 4)}

In [7]:
# Numerical-mode CPP: same algorithm and df_feat schema as CPP.run
cpp = aa.CPP(df_parts=df_parts_num,
             df_scales=df_scales_num,
             df_cat=df_cat_num,
             verbose=False)
df_feat = cpp.run_num(dict_num_parts=dict_num_parts,
                      labels=labels,
                      n_filter=10,
                      n_jobs=1)        # serial: avoids spawn footgun on Py3.14 + macOS
df_feat[["feature", "category", "subcategory", "mean_dif", "abs_auc"]].head()

,feature,category,subcategory,mean_dif,abs_auc
0,"TMD_C_JMD_C-Pattern(C,4,7,10)-dim_1",Embeddings,Embeddings_cat1_subcat0,-0.216,0.36
1,"TMD_C_JMD_C-Pattern(N,5,9,12,15)-dim_1",Embeddings,Embeddings_cat1_subcat0,-0.174,0.36
2,"JMD_N_TMD_N-Pattern(N,3,6,10,14)-dim_0",Embeddings,Embeddings_cat0_subcat1,0.174,0.36
3,"JMD_N_TMD_N-Pattern(C,5,8,11,14)-dim_3",Embeddings,Embeddings_cat0_subcat2,0.165,0.35
4,"TMD-Pattern(N,2,5,8)-dim_3",Embeddings,Embeddings_cat0_subcat2,0.152,0.34


In [8]:
# Multiple per-residue sources combine along the D axis before get_parts.
# Here a second synthetic channel stands in for a structural tensor.
dict_struct = {e: rng.random((len(s), 2)) for e, s in zip(df_seq["entry"], df_seq["sequence"])}
dict_combined = aa.combine_dict_nums([dict_num, dict_struct])
{e: dict_combined[e].shape for e in list(dict_combined)[:3]}   # D = 4 + 2 = 6

{'Q14802': (87, 6), 'Q86UE4': (582, 6), 'Q969W9': (287, 6)}

## Output

**Arm A (sequence):**
- `df_parts` - one row per protein, one column per part (`tmd`, `jmd_n_tmd_n`, `tmd_c_jmd_c`); the primary input to `CPP`.
- `split_kws` - the split configuration dict.
- `features` - a list of `PART-SPLIT-SCALE` feature ids, optionally reduced by `filter_correlation`.

**Arm B (numerical):**
- `(df_parts_num, dict_num_parts)` from `get_parts`; each `dict_num_parts[part]` is a NaN-padded array of shape `(n_samples, L_part_max, D)` aligned row-for-row with `df_parts_num`.
- `df_feat` - the signature, with the **same schema** as `CPP.run` (`feature`, `category`, `subcategory`, `mean_dif`, `abs_auc`, ...). The value axis is now the embedding dimensions (`dim_0` ... `dim_3`) instead of physicochemical scales.

## How to interpret

| Output | Reading |
| --- | --- |
| a `df_parts` row | the default parts (the base part `tmd` plus the composites `jmd_n_tmd_n` and `tmd_c_jmd_c`, derived from the base parts `jmd_n`/`tmd`/`jmd_c`) of one protein |
| a `split_kws` entry | the rule for *which positions* are averaged (`Segment` = contiguous block, `Pattern`/`PeriodicPattern` = spaced positions) |
| a `PART-SPLIT-SCALE` id (Arm A) | *where* + *how* + *which physicochemical property* |
| a `PART-SPLIT-dimension` id (Arm B) | *where* + *how* + *which embedding/structure channel* (`dim_0` ...) |
| `mean_dif` sign | direction of separation (positive = higher in the test class in that region) |
| `abs_auc` | effect size / separation strength of the feature |

Both arms read the same way: a coherent block of high-`abs_auc` features from one property family or embedding channel, localised to one part, is what distinguishes substrates from non-substrates. The numerical arm simply lets you profile learned/structural representations with the same interpretable grammar. One caveat for this demo: because the Arm B tensor is synthetic NumPy noise, the `abs_auc`/`mean_dif` values shown above are **not** biologically meaningful (the small effect sizes are pure chance). Swap in a real ProtT5/ESM embedding or a DSSP/structure tensor of the same `(L, D)` shape and the identical readout surfaces the channels that actually separate substrates from non-substrates.

## Common mistakes

- **Passing `df_seq` into `CPP`** - `CPP` takes `df_parts`; build them with `get_df_parts` (Arm A) or `get_parts` (Arm B) first.
- **Feeding raw, unbounded embeddings into `run_num`** - values must be in `[0, 1]` (the `max_std_test` pre-filter is calibrated for that range); normalise or encode first. Our synthetic tensor is already in range.
- **`dict_num` length mismatch** - each `dict_num[entry]` must have `L == len(sequence)`; otherwise `get_parts` cannot align the tensor to the parts.
- **`D` mismatch** - `D` must equal `len(df_scales.columns)` and `len(df_cat)`; deriving both with `build_scales`/`build_cat` from the same `dict_num` keeps the invariant.
- **Using `len(df_seq)` for class sizes** - use the `label` column; `load_dataset(..., n=N)` returns `2N` rows.
- **Relying on the default `n_jobs`** - on Python 3.14 + macOS, multiprocessing spawn without a `__main__` guard is fragile; pass `n_jobs=1` in notebooks.
- **Enumerating the full scale set** - `get_features` over hundreds of scales explodes combinatorially; slice `list_scales` for exploration.

## Next step

- **Compositional vs positional** - see *Protocol 5 - Compositional vs positional*, which shows how `split_kws` controls the trade-off between composition-like features (whole-part `Segment(1,1)`) and position-resolved features (`Segment` with `n_split_max > 1`, `Pattern`, `PeriodicPattern`).
- **Run the signature** - feed the engineered `df_parts` into the CPP signature: see *Protocol 1 - CPP signature*.